# ConvNeXt Unet satellite

# 1. SETUP AND IMPORTS

In [1]:
!pip install rasterio torchinfo "wandb[media]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 57.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
preprocessing 0.1.13 requires nltk==3.2.4, but you have nltk 3.9.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 2.8.0 requires google-cloud-bigquery[bqstorage,pandas]>=3.31.0, but you have google-cloud-bigquery 3.25.0 which is incompatible.
bigframes 2.8.0 requires r

In [2]:
# Standard Library Imports
import os
import time
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict

# Data Handling and Visualization
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import rasterio
from rasterio.errors import NotGeoreferencedWarning

# PyTorch and Torchvision
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# TorchMetrics for robust evaluation
import torchmetrics
from torchmetrics.classification import BinaryJaccardIndex

# Utilities
from tqdm.notebook import tqdm
from torchinfo import summary

# Experiment Tracking
import wandb

# Ignore non-critical rasterio warnings about georeferencing
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

In [3]:
# --- W&B LOGIN ---
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=api_key)
    print("W&B login successful from Kaggle Secrets.")
except ImportError:
    print("Proceeding with standard W&B login. Please enter your API key if prompted.")
    wandb.login()
except Exception as e:
    print(f"Fatal error: Could not log in to W&B. Please check your Kaggle Secret. Error: {e}")
    # raise e # Commented out to allow offline running if desired

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: duncanb013 (duncanb013-polytechnic-university-of-the-philippines) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful from Kaggle Secrets.


# 2. CONFIGURATION

Adjusted for Single Modality (Satellite RGB).

In [4]:
@dataclass
class Config:
    """
    Central configuration class for the training pipeline.
    """
    # --- Data Paths ---
    ORIGINAL_DATA_ROOT: Path = Path("/kaggle/input/quezon-city-informal-settlements/qc")
    AUGMENTED_DATA_ROOT: Path = Path("/kaggle/input/quezon-city-informal-settlements/qc-aug")
    SAT_SLIDEMIX_AUG_DATA_ROOT: Path = Path("/kaggle/input/quezon-city-informal-settlements/qc-sat-slidemix-aug")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/")

    # --- Experiment Settings ---
    # MODALITY_TO_RUN is fixed to "satellite" for this notebook version
    MODALITY_TO_RUN: str = "satellite"
    INPUT_CHANNELS: int = 3
    
    # --- Data Splitting and Loading ---
    VAL_SPLIT: float = 0.25
    TEST_SPLIT: float = 0.25
    RANDOM_SEED: int = 42
    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 2

    # --- Model Architecture (ConvNeXt Encoder) ---
    # Standard ConvNeXt-Tiny style dims: [96, 192, 384, 768] (adjusted slightly for custom needs)
    # Using: [80, 160, 320, 640]
    ENCODER_CHANNEL_LIST: list[int] = field(default_factory=lambda: [80, 160, 320, 640])
    ENCODER_BLOCKS_PER_STAGE: list[int] = field(default_factory=lambda: [2, 2, 8, 2]) # [3, 3, 9, 3] is standard Tiny
    
    # --- UNetDecoder Specific ---
    UNET_DECODER_CHANNEL_LIST: list[int] = field(default_factory=lambda: [512, 256, 128, 64])
    FINAL_UPSAMPLING_CHANNELS: list[int] = field(default_factory=lambda: [64]) # Just one final output scale

    # --- Hyperparams ---
    ENCODER_DROP_PATH_RATE: float = 0.0
    ENCODER_LAYER_SCALE_INIT_VALUE: float = 1e-6
    
    # --- Training Parameters ---
    NUM_EPOCHS: int = 100
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 5e-4
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    # --- Augmentations & Normalization ---
    AUGMENTATION_PROB: float = 0.5
    
    # Dataset stats for RGB Satellite
    RGB_MEAN: list[float] = field(default_factory=lambda: [0.33969313, 0.35239491, 0.28135468])
    RGB_STD: list[float] = field(default_factory=lambda: [0.23594516, 0.20353660, 0.20314776])
    
    # --- Evaluation ---
    METRIC_THRESHOLD: float = 0.6 

    # --- Dynamic Properties ---
    CURRENT_MEAN: list[float] = field(default_factory=list)
    CURRENT_STD: list[float] = field(default_factory=list)

def setup_config() -> Config:
    """Initializes and updates the configuration."""
    config = Config()
    
    # Set Normalization parameters for RGB
    config.CURRENT_MEAN = config.RGB_MEAN
    config.CURRENT_STD = config.RGB_STD
    
    print(f"Running experiment with MODALITY: {config.MODALITY_TO_RUN} ({config.INPUT_CHANNELS} channels)")
    return config

# --- Initialize ---
config = setup_config()

# --- Reproducibility ---
torch.manual_seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)
random.seed(config.RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.RANDOM_SEED)

Running experiment with MODALITY: satellite (3 channels)


# 3. UTILITIES AND HELPER FUNCTIONS

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        if self.data_format not in ["channels_last", "channels_first"]:
            raise NotImplementedError
        self.normalized_shape = (normalized_shape,)

    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None] * x + self.bias[:, None, None]
            return x

def drop_path(x, drop_prob: float = 0., training: bool = False, scale_by_keep: bool = True):
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = x.new_empty(shape).bernoulli_(keep_prob)
    if keep_prob > 0.0 and scale_by_keep:
        random_tensor.div_(keep_prob)
    return x * random_tensor

class DropPath(nn.Module):
    def __init__(self, drop_prob=None, scale_by_keep=True):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob
        self.scale_by_keep = scale_by_keep

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training, self.scale_by_keep)

def analyze_mask_distribution(config: Config, id_splits: dict):
    print("\n--- Analyzing Mask Pixel Distribution ---")
    def _find_mask_path(img_stem, config):
        possible_exts = ['.png', '.jpg', '.tif', '.jpeg']
        # Prioritize original mask dir
        base_dir = config.ORIGINAL_DATA_ROOT / "mask-256"
        for ext in possible_exts:
            file_path = base_dir / (img_stem + ext)
            if file_path.exists():
                return file_path
        return None

    for split_name, ids in id_splits.items():
        if not ids: continue
        total_pixels = 0
        target_pixels = 0
        for img_id in tqdm(ids, desc=f"Analyzing {split_name} masks", leave=False):
            mask_path = _find_mask_path(img_id, config)
            if not mask_path: continue
            with Image.open(mask_path) as mask_img:
                mask_np = (np.array(mask_img.convert("L")) > 0)
                target_pixels += np.sum(mask_np)
                total_pixels += mask_np.size
        if total_pixels > 0:
            target_percent = (target_pixels / total_pixels) * 100
            print(f"Split: {split_name.upper():<5} | Tiles: {len(ids):<5} | Target: {target_percent:.2f}% | Background: {100-target_percent:.2f}%")
    print("-----------------------------------------")

# 4. DATA PIPELINE

Simplified for RGB Satellite imagery only.

In [6]:
class SatelliteDataset(Dataset):
    """
    Custom dataset for loading RGB satellite imagery and binary masks.
    """
    def __init__(self, config: Config, image_ids: list[str], transform=None, aug_transform=None):
        self.original_data_root = config.ORIGINAL_DATA_ROOT
        self.image_ids = image_ids
        self.transform = transform
        self.aug_transform = aug_transform
        self.possible_exts = ['.png', '.jpg', '.tif', '.jpeg']

        # Paths
        self.sat_dir = self.original_data_root / "satellite-256"
        self.mask_dir = self.original_data_root / "mask-256"

    def __len__(self):
        return len(self.image_ids)

    def _find_file(self, base_dir, file_stem):
        for ext in self.possible_exts:
            file_path = base_dir / (file_stem + ext)
            if file_path.exists():
                return file_path
        raise FileNotFoundError(f"Could not find file for ID '{file_stem}' in {base_dir}")

    def __getitem__(self, idx):
        img_base_name = self.image_ids[idx]
            
        # 1. Load Satellite Image (RGB)
        path = self._find_file(self.sat_dir, img_base_name)
        # Load as RGB, normalize to 0-1 range floats
        image_to_process_HWC = np.array(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0

        # 2. Load Mask
        mask_path = self._find_file(self.mask_dir, img_base_name)
        mask_np_HW = (np.array(Image.open(mask_path).convert("L")) > 0).astype(np.float32)

        # 3. Apply Augmentations (Albumentations)
        if self.aug_transform:
            augmented = self.aug_transform(image=image_to_process_HWC, mask=mask_np_HW)
            image_aug_HWC, mask_aug_HW = augmented['image'], augmented['mask']
        else:
            image_aug_HWC, mask_aug_HW = image_to_process_HWC, mask_np_HW

        # 4. Apply Final Transforms (Normalization + ToTensor)
        if self.transform:
            transformed = self.transform(image=image_aug_HWC, mask=mask_aug_HW)
            image_CHW, mask_HW = transformed['image'], transformed['mask']
            mask_1HW = mask_HW.unsqueeze(0) # Add channel dim to mask
        else:
            # Fallback if no transform provided
            image_CHW = torch.from_numpy(image_aug_HWC.transpose(2, 0, 1))
            mask_1HW = torch.from_numpy(mask_aug_HW).unsqueeze(0).float()

        return image_CHW, mask_1HW

# --- Custom Augmentation ---
class SatSlideMixAlb(A.DualTransform):
    """
    Custom Albumentations transform: Circular shift along a random dimension.
    """
    def __init__(self, beta=(0.0, 1.0), p=0.5):
        super().__init__(p=p)
        self.beta = beta

    def get_params_dependent_on_data(self, params, data):
        height, width = data['image'].shape[:2]
        dims = {0: height, 1: width}
        dim = self.py_random.choice([0, 1])
        shift_fraction = self.py_random.uniform(self.beta[0], self.beta[1])
        size = dims[dim]
        shift = int(size * shift_fraction)
        return {"dim": dim, "shift": shift}

    def apply(self, img, dim=0, shift=0, **params):
        return np.roll(img, shift=shift, axis=dim)

    def apply_to_mask(self, mask, dim=0, shift=0, **params):
        return np.roll(mask, shift=shift, axis=dim)

    def get_transform_init_args_names(self):
        return ("beta",)


def get_transforms(config: Config, is_train=True):
    aug_transform = None
    if is_train:
        aug_transform = A.Compose([
            A.HorizontalFlip(p=config.AUGMENTATION_PROB),
            A.VerticalFlip(p=config.AUGMENTATION_PROB),
            A.RandomRotate90(p=config.AUGMENTATION_PROB),
            SatSlideMixAlb(p=config.AUGMENTATION_PROB)
        ])

    final_transform = A.Compose([
        A.Normalize(mean=config.CURRENT_MEAN, std=config.CURRENT_STD, max_pixel_value=1.0),
        ToTensorV2()
    ])
    return aug_transform, final_transform

def get_dataloaders(config: Config):
    # Determine IDs from satellite folder
    original_satellite_dir = config.ORIGINAL_DATA_ROOT / "satellite-256"
    all_base_ids = sorted([f.stem for f in original_satellite_dir.iterdir() if f.is_file()])
    
    total_size = len(all_base_ids)
    val_size = int(config.VAL_SPLIT * total_size)
    test_size = int(config.TEST_SPLIT * total_size)
    
    # Shuffle
    generator = torch.Generator().manual_seed(config.RANDOM_SEED)
    indices = torch.randperm(total_size, generator=generator).tolist()
    
    train_ids = [all_base_ids[i] for i in indices[:-val_size-test_size]]
    val_ids = [all_base_ids[i] for i in indices[-val_size-test_size:-test_size]]
    test_ids = [all_base_ids[i] for i in indices[-test_size:]]

    print("\n--- Dataset Splitting ---")
    print(f"Total: {total_size} | Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")

    train_aug, train_final = get_transforms(config, is_train=True)
    val_aug, val_final = get_transforms(config, is_train=False)

    dataloaders = {}
    if train_ids:
        train_ds = SatelliteDataset(config, train_ids, train_final, train_aug)
        dataloaders['train'] = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS, pin_memory=True)
    if val_ids:
        val_ds = SatelliteDataset(config, val_ids, val_final, val_aug)
        dataloaders['val'] = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=True)
    if test_ids:
        test_ds = SatelliteDataset(config, test_ids, val_final, val_aug)
        dataloaders['test'] = DataLoader(test_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=True)

    id_splits = {'train': train_ids, 'val': val_ids, 'test': test_ids}
    return dataloaders, id_splits

# 5. MODEL COMPONENTS

Components for ConvNeXt Encoder and Plain UNet Decoder.

In [7]:
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, drop_path_rate=0., layer_scale_init_value=1e-6):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = LayerNorm(dim, eps=1e-6, data_format="channels_first")
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)
        self.gamma = nn.Parameter(layer_scale_init_value * torch.ones((dim, 1, 1)), requires_grad=True) if layer_scale_init_value > 0 else None
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0. else nn.Identity()

    def forward(self, x):
        shortcut = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = shortcut + self.drop_path(x)
        return x

class ConvNeXtEncoder(nn.Module):
    """
    ConvNeXt Encoder that outputs feature maps from 4 stages (stem + 3 downsampling stages).
    """
    def __init__(self, config: Config, in_chans: int):
        super().__init__()
        self.dims = config.ENCODER_CHANNEL_LIST
        self.depths = config.ENCODER_BLOCKS_PER_STAGE
        
        total_blocks = sum(self.depths)
        dp_rates = [x.item() for x in torch.linspace(0, config.ENCODER_DROP_PATH_RATE, total_blocks)]
        
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(in_chans, self.dims[0], kernel_size=4, stride=4),
            LayerNorm(self.dims[0], eps=1e-6, data_format="channels_first")
        )
        
        self.stages = nn.ModuleList()
        self.downsamplers = nn.ModuleList()
        
        cursor = 0
        for i in range(4):
            # Downsamplers
            if i > 0:
                downsampler = nn.Sequential(
                    LayerNorm(self.dims[i-1], eps=1e-6, data_format="channels_first"),
                    nn.Conv2d(self.dims[i-1], self.dims[i], kernel_size=2, stride=2),
                )
                self.downsamplers.append(downsampler)

            # Stage blocks
            stage_dp_rates = dp_rates[cursor : cursor + self.depths[i]]
            stage = nn.Sequential(*[
                ConvNeXtBlock(self.dims[i], stage_dp_rates[j], config.ENCODER_LAYER_SCALE_INIT_VALUE)
                for j in range(self.depths[i])
            ])
            self.stages.append(stage)
            cursor += self.depths[i]
            
        self.output_channels = self.dims

    def forward(self, x):
        features = {}
        # Stem + Stage 0
        x = self.stem(x)
        x = self.stages[0](x)
        features['s1'] = x
        
        # Stage 1
        x = self.downsamplers[0](x)
        x = self.stages[1](x)
        features['s2'] = x
        
        # Stage 2
        x = self.downsamplers[1](x)
        x = self.stages[2](x)
        features['s3'] = x
        
        # Stage 3 (Bottleneck equivalent)
        x = self.downsamplers[2](x)
        x = self.stages[3](x)
        features['s4'] = x 
        
        return features

class UNetDecoder(nn.Module):
    """
    Standard U-Net Decoder.
    Takes the feature dictionary from the encoder, performs upsampling, concatenation (skip connection),
    and standard double convolutions.
    """
    def __init__(self, config: Config, encoder_channels: list[int]):
        super().__init__()
        s1_ch, s2_ch, s3_ch, s4_ch = encoder_channels
        d4_ch, d3_ch, d2_ch, d1_ch = config.UNET_DECODER_CHANNEL_LIST

        def double_conv(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
            )

        # Bottleneck processing (s4)
        self.bottleneck = double_conv(s4_ch, s4_ch)

        # Decoder Stage 1 (processing s3)
        self.up1 = nn.ConvTranspose2d(s4_ch, d4_ch, kernel_size=2, stride=2)
        self.dec_block1 = double_conv(d4_ch + s3_ch, d4_ch)

        # Decoder Stage 2 (processing s2)
        self.up2 = nn.ConvTranspose2d(d4_ch, d3_ch, kernel_size=2, stride=2)
        self.dec_block2 = double_conv(d3_ch + s2_ch, d3_ch)

        # Decoder Stage 3 (processing s1)
        self.up3 = nn.ConvTranspose2d(d3_ch, d2_ch, kernel_size=2, stride=2)
        self.dec_block3 = double_conv(d2_ch + s1_ch, d2_ch)

        # Decoder Stage 4 (Up to original resolution / 4)
        self.up4 = nn.ConvTranspose2d(d2_ch, d1_ch, kernel_size=2, stride=2)
        self.dec_block4 = double_conv(d1_ch, d1_ch)

        # Final Upsampling (4x to match input resolution)
        # Note: ConvNeXt stem is 4x downsample. S1 is 4x.
        # We need to reach original resolution. 
        # S1 is 1/4 res. dec_block3 output is 1/4 res (d2_ch).
        # up4 goes to 1/2 res? No, encoder steps are usually 4x, 8x, 16x, 32x.
        # s1=1/4, s2=1/8, s3=1/16, s4=1/32.
        # up1(s4) -> 1/16. + s3.
        # up2 -> 1/8. + s2.
        # up3 -> 1/4. + s1.
        # up4 -> 1/2.
        # final_up -> 1/1.
        
        self.final_up = nn.ConvTranspose2d(d1_ch, config.FINAL_UPSAMPLING_CHANNELS[-1], kernel_size=2, stride=2)
        self.final_conv_out = nn.Conv2d(config.FINAL_UPSAMPLING_CHANNELS[-1], 1, kernel_size=1)

    def forward(self, features: dict):
        s1, s2, s3, s4 = features['s1'], features['s2'], features['s3'], features['s4']

        b = self.bottleneck(s4)
        
        x = self.up1(b)
        x = torch.cat([x, s3], dim=1) # Skip connection
        x = self.dec_block1(x)

        x = self.up2(x)
        x = torch.cat([x, s2], dim=1) # Skip connection
        x = self.dec_block2(x)

        x = self.up3(x)
        x = torch.cat([x, s1], dim=1) # Skip connection
        x = self.dec_block3(x)
        
        x = self.up4(x)
        x = self.dec_block4(x)
        
        x = self.final_up(x)
        return self.final_conv_out(x)

# 6. MODEL ASSEMBLY

In [8]:
class ConvNeXtUNet_PlainDecoder(nn.Module):
    """
    Architecture: ConvNeXt Encoder + Plain UNet Decoder
    Input: Single Modality (Satellite RGB)
    """
    def __init__(self, config: Config):
        super().__init__()
        self.encoder = ConvNeXtEncoder(config, in_chans=config.INPUT_CHANNELS)
        self.decoder = UNetDecoder(config, encoder_channels=self.encoder.output_channels)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

    def forward(self, x):
        features = self.encoder(x)
        return self.decoder(features)

# 7. LOSS FUNCTIONS

In [9]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        intersection = (probs_flat * targets_flat).sum()
        dice_score = (2. * intersection + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)
        return 1 - dice_score

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, pos_weight=None):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce_with_logits = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_weight)

    def forward(self, logits, targets):
        bce_loss = self.bce_with_logits(logits, targets)
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_weight = (1 - pt).pow(self.gamma)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        focal_loss = alpha_t * focal_weight * bce_loss
        return focal_loss.mean()

class CombinedLoss(nn.Module):
    def __init__(self, focal_weight=0.5, dice_weight=0.5, focal_alpha=0.25, focal_gamma=2.0, pos_weight=None):
        super(CombinedLoss, self).__init__()
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight
        self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma, pos_weight=pos_weight)
        self.dice_loss = DiceLoss()

    def forward(self, logits, targets):
        loss_focal = self.focal_loss(logits, targets)
        loss_dice = self.dice_loss(logits, targets)
        return self.focal_weight * loss_focal + self.dice_weight * loss_dice

# 8. TRAINING, VALIDATION, RUN_EXPERIMENT

In [10]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, iou_metric):
    model.train(); total_loss = 0.0; iou_metric.reset()
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    for images, masks in progress_bar:
        images, masks = images.to(device), masks.to(device); optimizer.zero_grad()
        logits = model(images); loss = criterion(logits, masks); loss.backward(); optimizer.step()
        total_loss += loss.item(); preds = torch.sigmoid(logits).detach(); iou_metric.update(preds, masks)
        progress_bar.set_postfix(loss=loss.item())
    return total_loss / len(dataloader), iou_metric.compute().item()

def validate_one_epoch(model, dataloader, criterion, device, iou_metric):
    model.eval(); total_loss = 0.0; iou_metric.reset()
    progress_bar = tqdm(dataloader, desc="Validating", leave=False)
    with torch.no_grad():
        for images, masks in progress_bar:
            images, masks = images.to(device), masks.to(device)
            logits = model(images); loss = criterion(logits, masks)
            total_loss += loss.item(); preds = torch.sigmoid(logits); iou_metric.update(preds, masks)
            progress_bar.set_postfix(loss=loss.item())
    return total_loss / len(dataloader), iou_metric.compute().item()

def evaluate_on_test_set(model, dataloader, device, threshold=0.5):
    model.eval()
    metric_collection = torchmetrics.MetricCollection({
        'Accuracy': torchmetrics.classification.BinaryAccuracy(threshold=threshold),
        'Precision': torchmetrics.classification.BinaryPrecision(threshold=threshold),
        'Recall': torchmetrics.classification.BinaryRecall(threshold=threshold),
        'F1Score': torchmetrics.classification.BinaryF1Score(threshold=threshold),
        'IoU (Jaccard)': torchmetrics.classification.BinaryJaccardIndex(threshold=threshold),
        'ConfusionMatrix': torchmetrics.classification.BinaryConfusionMatrix(threshold=threshold)
    }).to(device)

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc=f"Evaluating (thresh={threshold})"):
            images, masks = images.to(device), masks.to(device)
            logits = model(images)
            preds = torch.sigmoid(logits)
            metric_collection.update(preds, masks)
            
    final_metrics = metric_collection.compute()
    
    # Log Confusion Matrix to W&B
    cm_tensor = final_metrics.pop('ConfusionMatrix', None)
    if wandb.run and cm_tensor is not None:
        wandb.log({
            f"test/confusion_matrix_at_{threshold}": wandb.plot.confusion_matrix(
                matrix_values=cm_tensor.cpu().numpy(),
                class_names=["Background", "Settlement"],
                title=f"Confusion Matrix @ Threshold {threshold}"
            )
        })

    print(f"\n--- Final Metrics (Threshold: {threshold}) ---")
    wandb_summary_metrics = {}
    for name, value in final_metrics.items():
        print(f"{name:<15}: {value.item():.4f}")
        wandb_summary_metrics[f"test/{name.lower()}_at_{threshold}"] = value.item()
        
    if wandb.run:
        wandb.summary.update(wandb_summary_metrics)


def log_predictions_to_wandb(model, dataloader, config, num_samples=32, prediction_threshold=0.5):
    if not dataloader or not wandb.run: return
    print(f"\n--- Logging Prediction Examples to W&B (Threshold: {prediction_threshold}) ---")
    model.eval()
    dataset = dataloader.dataset
    torch.manual_seed(42) 
    indices = torch.randperm(len(dataset))[:num_samples]
    
    wandb_images = []
    mean = torch.tensor(config.CURRENT_MEAN, device=config.DEVICE).view(-1, 1, 1)
    std = torch.tensor(config.CURRENT_STD, device=config.DEVICE).view(-1, 1, 1)

    with torch.no_grad():
        for idx in tqdm(indices, desc="Generating W&B Images"):
            image_tensor, mask_tensor = dataset[idx]
            image_tensor = image_tensor.unsqueeze(0).to(config.DEVICE)
            
            logits = model(image_tensor)
            pred_probs = torch.sigmoid(logits)
            pred_mask = (pred_probs > prediction_threshold).float()
            
            # De-normalize image for visualization
            denorm_image = image_tensor[0] * std + mean
            img_np = np.clip(denorm_image.cpu().permute(1, 2, 0).numpy(), 0, 1)
            
            # Create Plot
            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(img_np); axes[0].set_title('Satellite RGB'); axes[0].axis('off')
            axes[1].imshow(mask_tensor.squeeze(), cmap='gray'); axes[1].set_title('Ground Truth'); axes[1].axis('off')
            axes[2].imshow(pred_mask.cpu().squeeze(), cmap='gray'); axes[2].set_title('Prediction'); axes[2].axis('off')

            plt.tight_layout()
            wandb_images.append(wandb.Image(fig, caption=f"ID: {dataset.image_ids[idx]}"))
            plt.close(fig)
            
    wandb.log({"test_predictions": wandb_images})

def run_experiment(config, model, dataloaders, model_name):
    device = torch.device(config.DEVICE)
    model.to(device)
    # Weighted loss for class imbalance (approx 24:1 bg:target)
    pos_weight_tensor = torch.tensor([24.0], device=device)
    criterion = CombinedLoss(focal_weight=0.5, dice_weight=0.5, pos_weight=pos_weight_tensor).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    train_iou_metric = BinaryJaccardIndex(threshold=config.METRIC_THRESHOLD).to(device)
    val_iou_metric = BinaryJaccardIndex(threshold=config.METRIC_THRESHOLD).to(device)
    
    history = {'train_loss': [], 'train_iou': [], 'val_loss': [], 'val_iou': []}

    with wandb.init(
        project="settlement-segmentation-single-modality", 
        name=f"{model_name}_RGB_{int(time.time())}",
        config=asdict(config)
    ) as run:
    
        print("\n" + "="*80)
        print(f"MODEL: {model_name}")
        print("="*80)
        print(summary(model, input_size=(config.BATCH_SIZE, config.INPUT_CHANNELS, 256, 256), depth=4))
        print("\n" + "="*80)
        
        wandb.watch(model, criterion, log="all", log_freq=100)
        best_val_iou = 0.0
        model_path = config.CHECKPOINT_DIR / "best_model.pth"

        print(f"--- Starting Training for {config.NUM_EPOCHS} epochs on {device} ---")
        for epoch in range(config.NUM_EPOCHS):
            start_time = time.time();
            train_loss, train_iou = train_one_epoch(model, dataloaders['train'], criterion, optimizer, device, train_iou_metric)
            val_loss, val_iou = validate_one_epoch(model, dataloaders['val'], criterion, device, val_iou_metric)
            history['train_loss'].append(train_loss); history['train_iou'].append(train_iou)
            history['val_loss'].append(val_loss); history['val_iou'].append(val_iou)
            
            print(f"Epoch {epoch+1}/{config.NUM_EPOCHS} -> Train Loss: {train_loss:.4f}, IoU: {train_iou:.4f} | Val Loss: {val_loss:.4f}, IoU: {val_iou:.4f}")
            wandb.log({"epoch": epoch + 1, "train/loss": train_loss, "train/iou": train_iou, "val/loss": val_loss, "val/iou": val_iou})

            if val_iou > best_val_iou:
                best_val_iou = val_iou
                torch.save(model.state_dict(), model_path)
                wandb.summary["best_val_iou"] = best_val_iou
        
        if model_path.exists():
            print("\nLogging best model to W&B...")
            artifact = wandb.Artifact(name=f"{run.id}-best-model", type="model", metadata=asdict(config))
            artifact.add_file(model_path); run.log_artifact(artifact)
            
            print("\n--- Running Inference on Test Set ---")
            model.load_state_dict(torch.load(model_path))
            if dataloaders.get('test'):
                for threshold in [0.5, 0.6]:
                    evaluate_on_test_set(model, dataloaders['test'], config.DEVICE, threshold)
                log_predictions_to_wandb(model, dataloaders.get('test'), config, prediction_threshold=config.METRIC_THRESHOLD)

    return history

def plot_history(history):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.plot(history['train_loss'], label='Train'); plt.plot(history['val_loss'], label='Val'); plt.title('Loss'); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(history['train_iou'], label='Train'); plt.plot(history['val_iou'], label='Val'); plt.title('IoU'); plt.legend(); plt.show()

# 9. MAIN EXECUTION

In [11]:
# --- 1. Get Dataloaders ---
dataloaders, id_splits = get_dataloaders(config)

# --- 2. Select Model (ConvNeXt Encoder + Plain UNet Decoder) ---
model_name = "ConvNeXtUNet_PlainDecoder"
model = ConvNeXtUNet_PlainDecoder(config)

print(f"\n--- Model Selected: {model_name} ---")
print(f"Input Channels: {config.INPUT_CHANNELS} (Satellite RGB)")

# --- 3. Run Experiment ---
history = run_experiment(config, model, dataloaders, model_name)

# --- 4. Plot Results ---
plot_history(history)


--- Dataset Splitting ---
Total: 10005 | Train: 5003 | Val: 2501 | Test: 2501

--- Model Selected: ConvNeXtUNet_PlainDecoder ---
Input Channels: 3 (Satellite RGB)


wandb: Tracking run with wandb version 0.20.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20251213_145027-bnuk8ma5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ConvNeXtUNet_PlainDecoder_RGB_1765637427
wandb: ⭐️ View project at https://wandb.ai/duncanb013-polytechnic-university-of-the-philippines/settlement-segmentation-single-modality
wandb: 🚀 View run at https://wandb.ai/duncanb013-polytechnic-university-of-the-philippines/settlement-segmentation-single-modality/runs/bnuk8ma5



MODEL: ConvNeXtUNet_PlainDecoder
Layer (type:depth-idx)                        Output Shape              Param #
ConvNeXtUNet_PlainDecoder                     [64, 1, 256, 256]         --
├─ConvNeXtEncoder: 1-1                        [64, 640, 8, 8]           --
│    └─Sequential: 2-1                        [64, 80, 64, 64]          --
│    │    └─Conv2d: 3-1                       [64, 80, 64, 64]          3,920
│    │    └─LayerNorm: 3-2                    [64, 80, 64, 64]          160
│    └─ModuleList: 2-8                        --                        (recursive)
│    │    └─Sequential: 3-3                   [64, 80, 64, 64]          --
│    │    │    └─ConvNeXtBlock: 4-1           [64, 80, 64, 64]          55,840
│    │    │    └─ConvNeXtBlock: 4-2           [64, 80, 64, 64]          55,840
│    └─ModuleList: 2-7                        --                        (recursive)
│    │    └─Sequential: 3-4                   [64, 160, 32, 32]         --
│    │    │    └─LayerNorm: 4-3

Training:   0%|          | 0/79 [00:00<?, ?it/s]

Validating:   0%|          | 0/40 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch 100/100 -> Train Loss: 0.2392, IoU: 0.5377 | Val Loss: 0.3450, IoU: 0.3820

Logging best model to W&B...

--- Running Inference on Test Set ---


Evaluating (thresh=0.5):   0%|          | 0/40 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/tmp/ipykernel_19/2699937030.py", line 153, in run_experiment
    evaluate_on_test_set(model, dataloaders['test'], config.DEVICE, threshold)
  File "/tmp/ipykernel_19/2699937030.py", line 46, in evaluate_on_test_set
    f"test/confusion_matrix_at_{threshold}": wandb.plot.confusion_matrix(
                                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: confusion_matrix() got an unexpected keyword argument 'matrix_values'
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-summary.json
wandb: uploading summary, console lines 206-212
wandb:                                                                                
wandb: 
wandb: Run history:
wandb:      epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
wandb:  train/iou ▁▁▂▂▃▃▄▄▄▃▄▄▄▄▄▅▅▅▅▅▆▆▅▆▆▆▆▇▇▇▆▇▇▇▇▇▇▇▇█
wandb: train/loss █▇▅▅▃▃▃▃▃▃▃▃▃▃▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▁▁
wandb:    val/iou ▁▅▆▅▆▇▇▇▇▇▃▇▇▇▇▅▇▇▇▆▇█▆██▇█▇█▇█▇▇███▇██▇
wandb:   val/loss 

TypeError: confusion_matrix() got an unexpected keyword argument 'matrix_values'